Step 1: Detect comment language

In [2]:
import pandas as pd
from langdetect import detect, DetectorFactory

# Make language detection reproducible (langdetect is non-deterministic by default)
DetectorFactory.seed = 0

def detect_language(text):
    """Detect the language of a comment. Returns 'en', 'de', or 'unknown'."""
    if pd.isna(text) or not str(text).strip():
        return "unknown"
    try:
        return detect(str(text))
    except Exception:
        # Very short or ambiguous comments can fail detection
        return "unknown"

# Load the cleaned comments
df = pd.read_csv("../data/cleaned_comments.csv")

# Create the new language column
df["comment_language"] = df["Comment"].apply(detect_language)

# Display the language distribution
print("Language distribution:")
print(df["comment_language"].value_counts())

Language distribution:
comment_language
de         993
en         415
hu          50
ca          28
no          15
da          15
sv          14
nl          11
af          10
vi           8
tl           7
ro           7
id           6
pt           6
fi           4
it           4
unknown      3
es           2
so           2
pl           2
hr           2
sk           2
cs           1
et           1
sl           1
fr           1
Name: count, dtype: int64


Sanity Check

In [3]:
# Display 5 German and 5 English comments to verify detection
print("\n=== Detected as GERMAN ===")
print(df[df["comment_language"] == "de"]["Comment"].head())
print("\n=== Detected as ENGLISH ===")
print(df[df["comment_language"] == "en"]["Comment"].head())


=== Detected as GERMAN ===
0                                   best. LT 15.08.2025
17    RoHs issue, Taskforce sind wöchentliche Liefer...
18                                    Vorzug angefragt 
21    Unterlieferant in den Sommerferien Fehlmateria...
23                    Situation geprüft, nicht kritisch
Name: Comment, dtype: object

=== Detected as ENGLISH ===
1    on 23.03.2026 5.210pcs in quality inspection s...
2    1.620 pcs in quality inspection/to check the r...
3    Request sent out to both suppliers (Celestica ...
4     Request sent out to both suppliers (Celestica...
5             No stockout - Dashboard shows wrong data
Name: Comment, dtype: object


Step 2: Test Deepl API connection

In [ ]:
import deepl
from config import DEEPL_API_KEY

# Initialize the DeepL translator
translator = deepl.Translator(DEEPL_API_KEY)

# Test with a simple German sentence
result = translator.translate_text("Lieferverzögerung beim Lieferanten", target_lang="EN-US")
print("Translation test:", result.text)

# Check remaining character quota (Free plan)
usage = translator.get_usage()
print(f"\nCharacters used: {usage.character.count}")
print(f"Character limit: {usage.character.limit}")

Translation test: Delivery delay on the supplier's end

Characters used: 0
Character limit: 1000000


Step 3: Translate DE comment in EN using Deepl

In [13]:
import deepl
import pandas as pd
import time
from tqdm.notebook import tqdm
from config import DEEPL_API_KEY

# --- 1. Load the cleaned data with language column ---
df = pd.read_csv("../data/cleaned_comments.csv")
# Recreate the language column if not saved yet (skip if already in the CSV)
if "comment_language" not in df.columns:
    from langdetect import detect, DetectorFactory
    DetectorFactory.seed = 0
    def detect_language(text):
        if pd.isna(text) or not str(text).strip():
            return "unknown"
        try:
            return detect(str(text))
        except Exception:
            return "unknown"
    df["comment_language"] = df["Comment"].apply(detect_language)

# --- 2. Initialize the translator ---
translator = deepl.Translator(DEEPL_API_KEY)

# --- 3. Create the translation column (copy of original by default) ---
df["Comment_EN"] = df["Comment"]

# --- 4. Translate only German comments, in batches ---
german_mask = df["comment_language"] == "de"
german_comments = df.loc[german_mask, "Comment"].astype(str).tolist()

print(f"Number of German comments to translate: {len(german_comments)}")

BATCH_SIZE = 50
translations = []

for i in tqdm(range(0, len(german_comments), BATCH_SIZE), desc="Translating"):
    batch = german_comments[i : i + BATCH_SIZE]
    try:
        results = translator.translate_text(batch, target_lang="EN-US")
        translations.extend([r.text for r in results])
    except deepl.TooManyRequestsException:
        # Rate limit hit: wait 5 seconds and retry this batch
        print("Rate limit reached, waiting 5 seconds...")
        time.sleep(5)
        results = translator.translate_text(batch, target_lang="EN-US")
        translations.extend([r.text for r in results])

    # Checkpoint: save progress every 5 batches
    if (i // BATCH_SIZE) % 5 == 4:
        partial = df.loc[german_mask, "Comment_EN"].copy()
        # Save only what is translated so far
        print(f"Checkpoint at batch {i // BATCH_SIZE + 1}...")

# --- 5. Assign the translations back to the DataFrame ---
df.loc[german_mask, "Comment_EN"] = translations

# --- 6. Save the result ---
df.to_csv("../data/cleaned_comments.csv", index=False)
print("\n✅ Translation complete! Saved to data/cleaned_comments.csv")

# --- 7. Verify ---
print(f"\nCharacters used: {translator.get_usage().character.count}")
print("\n=== Before / After examples ===")
sample = df[df["comment_language"] == "de"][["Comment", "Comment_EN"]].head(5)
for _, row in sample.iterrows():
    print(f"DE: {row['Comment']}")
    print(f"EN: {row['Comment_EN']}\n")

Number of German comments to translate: 993


Translating:   0%|          | 0/20 [00:00<?, ?it/s]

Checkpoint at batch 5...
Checkpoint at batch 10...
Checkpoint at batch 15...
Checkpoint at batch 20...

✅ Translation complete! Saved to data/cleaned_comments.csv

Characters used: 223940

=== Before / After examples ===
DE: best. LT 15.08.2025
EN: best. LT August 15, 2025

DE: RoHs issue, Taskforce sind wöchentliche Lieferungen mit SVI am abklären
EN: RoHS issue: The task force is working with SVI to clarify the matter regarding weekly deliveries

DE: Vorzug angefragt 
EN: Request for Priority 

DE: Unterlieferant in den Sommerferien Fehlmaterial. Technisches Thema ebenfalls 
in Klärung via TSM.
Lieferant prüft die Fehlteile In House zu produzieren.
EN: Subcontractor: Missing materials during summer break. Technical issue also 
being clarified via TSM.
Supplier is considering producing the missing parts in-house.

DE: Situation geprüft, nicht kritisch
EN: Situation assessed; not critical

